# Tome 5, Chapter 10: Random models of the primes (Python companion)

Exercises first, solutions at the end. We flip Cramér's coins up to $10^7$ and compare with the primes (Theorem 10.2), hunt record gaps below $10^9$ (Theorem 10.4 and Figure 10.1), test the Hardy–Littlewood conjecture on prime constellations (Proposition 10.5), check Gallagher's average (Theorem 10.7), count primes in short intervals (Figure 10.2), watch the model get Goldbach wrong for odd numbers (Exercise 10.3), and finally use the density of primes to predict the cost of generating RSA keys. Requirements: numpy, sympy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★) Coins against primes.** Simulate three samples of Cramér's model up to $10^7$. For $x=10^3,\dots,10^7$, compare $\pi(x)-\operatorname{Li}(x)$ with $\pi_{\mathcal C}(x)-\operatorname{Li}(x)$ and with the bound $\sqrt{x\log x}$ of Theorem 10.2. Which is smaller, and is the sign of $\pi(x)-\operatorname{Li}(x)$ a surprise?

**P10.2 (★★) Record gaps.** Write a segmented sieve and list the record gaps between primes below $10^9$, with the ratio $\frac g{\log^2p}$. Find the largest gap in each Cramér sample below $10^7$. Who has larger gaps, the primes or the coins?

**P10.3 (★★) Constellations.** Implement $\nu_H(p)$, admissibility and the singular series $\mathfrak S(H)$. For several $H$, compare the number of $n\le10^7$ with all $n+h$ prime with the prediction $\mathfrak S(H)\int_2^x\frac{\mathrm dt}{\log^kt}$ (Exercise 10.2).

**P10.4 (★) Gallagher's average.** Compute $\sum_{d\le h}\mathfrak S(\{0,d\})-h$ for $h$ up to $10^5$. How does it compare with the bound $O(\sqrt h)$ of Theorem 10.7?

**P10.5 (★★) Short intervals.** Count the primes in a million consecutive intervals of length $h\approx\log N$ after $N=10^6$ and $N=10^9$. Compare the law with Poisson, and the variance with the Montgomery–Soundararajan prediction $\lambda\bigl(1-\frac{\log h+\gamma+\log2\pi-1}{\log N}\bigr)$.

**P10.6 (★) Goldbach in the model.** Count the representations $N=a+b$ with $3\le a\le\frac N2$ in a Cramér sample and in the primes, for a few even and odd $N$ near $10^6$.

**P10.7 (★★) CS application: generating RSA primes.** Key generation picks random odd $b$-bit integers until one passes a primality test. Use the prime number theorem to predict the expected number of tries, then measure it for $b=256$ and $512$ with `sympy.isprime`. What law should the number of tries follow?

## Solutions

### P10.1

In [ ]:
import math, time
import numpy as np
import mpmath as mp

rng = np.random.default_rng(2026)
N = 10**7
n = np.arange(N + 1, dtype=float)
prob = np.zeros(N + 1); prob[3:] = 1 / np.log(n[3:])
is_p = np.ones(N + 1, dtype=bool); is_p[:2] = False
for p in range(2, int(N**0.5) + 1):
    if is_p[p]: is_p[p*p::p] = False
Li = lambda x: float(mp.quad(lambda t: 1 / mp.log(t), [2, x]))
print("      x        pi(x)     Li(x)   pi-Li  |  three Cramér samples: pi_C - Li,  bound sqrt(x log x)")
samples = [rng.random(N + 1) < prob for _ in range(3)]
for k in range(3, 8):
    x = 10**k; li = Li(x)
    devs = "  ".join(f"{int(s[:x + 1].sum()) - li:8.0f}" for s in samples)
    print(f"{x:9d} {int(is_p[:x + 1].sum()):9d} {li:10.0f} {int(is_p[:x + 1].sum()) - li:6.0f}  |  {devs}   {math.sqrt(x * math.log(x)):8.0f}")

All the deviations are far below $\sqrt{x\log x}$: Hoeffding's inequality ignores that the variance is only about $\frac x{\log x}$, and the samples fluctuate on the scale $\sqrt{x/\log x}$, about $800$ at $10^7$ (the remark after Theorem 10.2). The primes do even better: $\pi(x)-\operatorname{Li}(x)$ is about $-340$ at $10^7$, and it is \emph{always negative} in this range, while the coins wander on both sides. The negative sign is the bias of Chapter 7: the squares of primes in $\psi$ push $\pi$ below $\operatorname{Li}$ by about $\frac{\sqrt x}{\log x}$. Littlewood proved that the sign does change, infinitely often; the first change is known to occur below $1.4\cdot10^{316}$.

### P10.2

In [ ]:
def record_gaps(flags):
    q = np.nonzero(flags)[0]
    g = np.diff(q)
    best, out = 0, []
    for i in np.nonzero(g == np.maximum.accumulate(g))[0]:
        if g[i] > best:
            best = int(g[i]); out.append((int(q[i]), best))
    return out

def segmented_primes_records(X, S=10**7):
    r = int(math.isqrt(X)) + 1
    base = np.ones(r + 1, bool); base[:2] = False
    for i in range(2, int(r**0.5) + 1):
        if base[i]: base[i*i::i] = False
    bp = np.nonzero(base)[0]
    last, best, out = None, 0, []
    for lo in range(0, X, S):
        seg = np.ones(min(S, X - lo), bool)
        if lo == 0: seg[:2] = False
        for p in bp:
            if p * p >= lo + len(seg): break
            st = max(p * p, -(-lo // p) * p)
            seg[st - lo::p] = False
        q = np.nonzero(seg)[0] + lo
        if last is not None: q = np.concatenate(([last], q))
        g = np.diff(q)
        for i in np.nonzero((g == np.maximum.accumulate(g)) & (g > best))[0]:
            if g[i] > best:
                best = int(g[i]); out.append((int(q[i]), best))
        last = int(q[-1])
    return out

t = time.time()
recs = segmented_primes_records(10**9)
print(f"record gaps between primes below 1e9 ({time.time() - t:.0f} s):")
for p, g in recs[-8:]:
    print(f"   p = {p:10d}  gap = {g:4d}   gap / log^2 p = {g / math.log(p)**2:.3f}")
for j, s in enumerate(samples):
    p, g = record_gaps(s)[-1]
    print(f"Cramér sample {j}: largest gap below 1e7 is {g} after {p}, gap / log^2 p = {g / math.log(p)**2:.3f}")
print("for comparison, log^2(1e7) =", round(math.log(1e7)**2, 1))

Below $10^9$ the ratio $\frac g{\log^2p}$ for the primes stays between $0.6$ and $0.75$; the Cramér samples, in a range a hundred times smaller, already reach $0.8$ and even $1.09$. At this height the primes have smaller record gaps than the coins. Heuristically, the small primes make the primes more evenly spread than independent coins, the effect that P10.5 measures in short intervals. Granville's refined model predicts the opposite at astronomical heights ($2e^{-\gamma}\log^2p$), and the record $1132$ (ratio $0.92$) shows the ratio does grow. Nobody knows its limit.

### P10.3

In [ ]:
from sympy import primerange
SMALL = list(primerange(2, 10**6))
def nu(H, p): return len({h % p for h in H})
def admissible(H): return all(nu(H, p) < p for p in primerange(2, len(H) + 1))
def S(H):
    k = len(H); s = 1.0
    for p in SMALL:
        v = nu(H, p)
        if v == p: return 0.0
        s *= (1 - v / p) / (1 - 1 / p)**k
    return s
def Ik(x, k): return float(mp.quad(lambda t: 1 / mp.log(t)**k, [2, 100, x]))
X = 10**7
pr = np.nonzero(is_p)[0]
isp = np.zeros(X + 40, bool); isp[:N + 1] = is_p
for H in ([0, 2], [0, 4], [0, 6], [0, 30], [0, 2, 4], [0, 2, 6], [0, 4, 6], [0, 2, 6, 8], [0, 2, 6, 8, 12]):
    c = int(np.sum(np.all([isp[pr + h] for h in H], axis=0) & (pr + max(H) <= X)))
    s = S(H)
    print(f"H = {str(H):17s} admissible: {admissible(H)!s:5s}  S(H) = {s:.4f}   count below 1e7: {c:7d}   predicted S*int dt/log^k t: {s * Ik(X, len(H)):9.0f}")

The agreement is remarkable: within $0.5\%$ for pairs, $1.5\%$ for triples, $4\%$ to $8\%$ for quadruples and quintuples, where the counts are small. Pairs at distance $6$ are twice as frequent as twins, and pairs at distance $30=2\cdot3\cdot5$ even more, $\frac{3-1}{3-2}\cdot\frac{5-1}{5-2}=\frac83$ times. The non-admissible $\{0,2,4\}$ occurs once, at $3,5,7$, where one of the three numbers \emph{is} the prime $3$. The Hardy–Littlewood conjecture has never failed a test of this kind.

### P10.4

In [ ]:
from sympy import factorint
C2 = S([0, 2]) / 2
def S2(d):
    if d % 2: return 0.0
    s = 2 * C2
    for p in factorint(d):
        if p > 2: s *= (p - 1) / (p - 2)
    return s
tot, h = 0.0, 0
for target in (10, 100, 1000, 10**4, 10**5):
    while h < target:
        h += 1; tot += S2(h)
    print(f"h = {h:6d}: sum_(d<=h) S(0,d) = {tot:10.3f},  minus h = {tot - h:7.3f},   -1/2 log h = {-0.5 * math.log(h):7.3f}")

The error is far smaller than $\sqrt h$: it follows $-\frac12\log h$ up to a roughly constant shift. The proof of Theorem 10.7 lost a lot in Rankin's trick, and a careful analysis of the Dirichlet series $\sum_m g(m)m^{-s}$ near $s=0$ gives the logarithm. This small negative term is exactly what reduces the variance in P10.5.

### P10.5

In [ ]:
def interval_counts(N0, h, M):
    L = h * M
    seg = np.ones(L, bool)
    for p in primerange(2, int(math.isqrt(N0 + L)) + 1):
        st = -(-(N0 + 1) // p) * p
        seg[st - (N0 + 1)::p] = False
    return seg.reshape(M, h).sum(1)
gam = 0.5772156649
for N0, h in ((10**6, 14), (10**9, 21)):
    c = interval_counts(N0, h, 10**6 if N0 > 10**6 else 50000)
    lam = c.mean(); lx = math.log(N0)
    ms = lam * (1 - (math.log(h) + gam + math.log(2 * math.pi) - 1) / lx)
    print(f"N = {N0:.0e}, h = {h}: mean {lam:.3f}, variance {c.var():.3f}, Poisson would say {lam:.3f}, Montgomery-Soundararajan {ms:.3f}")
    print("   P(k primes), k=0..5:", np.round(np.bincount(c, minlength=6)[:6] / len(c), 4))
    print("   Poisson          :", np.round([math.exp(-lam) * lam**k / math.factorial(k) for k in range(6)], 4))

The prime counts in short intervals have the right mean but a variance about $20\%$ to $30\%$ below the Poisson value: too few empty intervals, too few crowded ones, many more intervals with exactly one prime. The Montgomery–Soundararajan formula predicts the variance to two decimals at both heights. The correction is a factor $1-\frac{\log h+1.415}{\log N}$ that tends to $1$ only as $\frac1{\log N}$: Gallagher's Poisson law is the truth at infinity, and the singular series are the truth everywhere else.

### P10.6

In [ ]:
s = samples[0]
def reps(flags, M):
    a = np.arange(3, M // 2 + 1)
    return int(np.sum(flags[a] & flags[M - a]))
print("    N    model r(N)   primes r(N)")
for M in (10**6, 10**6 + 1, 10**6 + 2, 10**6 + 3, 2 * 10**6, 2 * 10**6 + 1):
    print(f"{M:8d} {reps(s, M):9d} {reps(is_p, M):12d}")
print("model prediction ~ N/(2 log^2 N):", round(10**6 / (2 * math.log(10**6)**2)))

The model sample writes every number near $10^6$ as a sum of two of its primes about $3000$ times, odd or even alike, close to $\frac N{2\log^2N}=2620$. The primes write the even ones $5000$ to $10\,000$ times, with large fluctuations that the singular series $2C_2\prod_{p\mid N,\,p>2}\frac{p-1}{p-2}$ explains ($10^6+2=2\cdot3\cdot166667$ is divisible by $3$, hence its large count), and the odd ones \emph{never}: one summand would have to be even, and our count starts at $a=3$, so the only candidate $2+(N-2)$ is excluded. This is Exercise 10.3 in numbers.

### P10.7

In [ ]:
from sympy import isprime
import random
random.seed(1)
for bits in (256, 512):
    tries = []
    for _ in range(200):
        t = 0
        while True:
            t += 1
            c = random.getrandbits(bits) | (1 << (bits - 1)) | 1
            if isprime(c): break
        tries.append(t)
    print(f"{bits}-bit odd candidates: mean tries {np.mean(tries):6.1f}, prediction (log 2^{bits})/2 = {bits * math.log(2) / 2:6.1f}, "
          f"std {np.std(tries):6.1f}, max {max(tries)}")

Near $2^b$, a random integer is prime with probability $\frac1{b\log2}$ and a random odd integer with probability $\frac2{b\log2}$, so we expect $\frac{b\log2}2$ tries: $89$ for $256$ bits and $177$ for $512$, matching the measurements. In a Cramér-style model the number of tries is geometric, so its standard deviation is about equal to its mean, which is what we see. In practice libraries first discard candidates with small prime factors, which multiplies the success rate by the factor $\prod_{p\le A}(1-\frac1p)^{-1}$ of Proposition 10.6, and use Miller–Rabin before any expensive test. The prime number theorem is a running-time estimate here.